## Generate Your Own data set


In [2]:
!pip install mediapipe


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 19.9 MB/s eta 0:00:00


In [20]:
from google.colab import drive
import os
import cv2
import json
import psycopg2
from tqdm import tqdm
import mediapipe as mp

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
conn = psycopg2.connect(
    database="asl",
    host="xxx",
    user="xxx",
    password="xxx",
    port=xxx,
)
cur = conn.cursor()
# Initialize MediaPipe
mp_drawing = mp.solutions.drawing_utils
pose_tools = mp.solutions.pose
pose_model = pose_tools.Pose()
hand_tools = mp.solutions.hands
hand_model = hand_tools.Hands()

# Path to videos in Google Drive
video_folder = "/content/drive/MyDrive/ASL/videos6/"
videos = os.listdir(video_folder)

bar = tqdm(total=len(videos))

try:
    for file in videos:
        bar.update(1)
        video_path = os.path.join(video_folder, file)

        word = file.split(".")[0]

        data = []
        frame_number = 0
        capture = cv2.VideoCapture(video_path)

        while capture.isOpened():
            success, frame = capture.read()
            if not success:
                break

            frame = cv2.resize(frame, (640, 480))
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            poses = pose_model.process(frame_rgb)
            hands = hand_model.process(frame_rgb)

            frame_number += 1
            point = [
                frame_number,
                [],
                [[], []],  # [left_hand_landmarks, right_hand_landmarks]
            ]

            if poses.pose_landmarks:
                for i, landmark in enumerate(poses.pose_landmarks.landmark):
                    point[1].append([i, landmark.x * 640, landmark.y * 480, landmark.z])

            if hands.multi_hand_landmarks:
                for landmarks in hands.multi_hand_landmarks:
                    hand = landmarks.landmark
                    handedness = 0 if (
                        hand[hand_tools.HandLandmark.WRIST].x
                        < hand[hand_tools.HandLandmark.THUMB_CMC].x
                    ) else 1

                    for i, landmark in enumerate(hand):
                        point[2][handedness].append(
                            [i, landmark.x * 640, landmark.y * 480, landmark.z]
                        )

            data.append(point)

        capture.release()

        try:
            cur.execute(
                "INSERT INTO sign (word, points) VALUES (%s, %s)",
                (word, json.dumps(data)),
            )
            conn.commit()
        except Exception as e:
            print(f"Database error: {e}")
            conn.rollback()

except Exception as e:
    print(f"Processing error: {e}")
finally:
    cur.close()
    conn.close()


100%|██████████| 6/6 [03:50<00:00, 38.37s/it]

 29%|██▊       | 2/7 [00:06<00:16,  3.37s/it]

Database error: duplicate key value violates unique constraint "sign_word_key"
DETAIL:  Key (word)=(love) already exists.




 43%|████▎     | 3/7 [00:12<00:16,  4.17s/it]

Database error: duplicate key value violates unique constraint "sign_word_key"
DETAIL:  Key (word)=(me) already exists.




 57%|█████▋    | 4/7 [00:17<00:13,  4.55s/it]

...

Database error: duplicate key value violates unique constraint "sign_word_key"
DETAIL:  Key (word)=(you) already exists.




 86%|████████▌ | 6/7 [00:28<00:05,  5.04s/it]

Database error: duplicate key value violates unique constraint "sign_word_key"
DETAIL:  Key (word)=(nice) already exists.




100%|██████████| 7/7 [00:30<00:00,  4.20s/it]

Database error: duplicate key value violates unique constraint "sign_word_key"
DETAIL:  Key (word)=(meet) already exists.

